# Iterative Cleaning with an Autoencoder
## Train on digits 0-6; inject digits 7-9

Train on mostly normal images, remove suspicious training images, then compare the results with controls.

- Normal digits: 0-6
- Injected digits: 7-9
- Labels are used for evaluation, not for training or removal.
- The score combines latent kNN distance and reconstruction error, as set by `SCORE_MODE`.

| Method | Description |
|---|---|
| Baseline | Train on the mixed data without removal. |
| Iterative | Remove a fraction, retrain, and repeat. |
| One-shot | Remove the same total count using the first-round scores. |
| Random | Remove the same total count at random. |
| Normal-only reference | Train on normal images only; this is a reference, not a removal method. |

Iterative, one-shot, and random removal use the same total removal budget.

**Metrics:** kNN accuracy on normal test digits; AUROC and average precision for anomaly detection. Run the cells in order.

## 1. Imports

In [11]:
import math                                   # math helpers (ceil etc.)
import random                                 # Python's random number generator (seeded for repeatability)
import time                                   # to time the run
from datetime import datetime                 # unique names for saved runs
from pathlib import Path                      # clean way to build file/folder paths

import numpy as np                            # arrays and numerical helpers
import torch                                  # PyTorch: tensors and neural networks
import torch.nn as nn                         # neural-network layers
import torch.nn.functional as F               # functions like mse_loss, one_hot, normalize
import matplotlib.pyplot as plt               # plotting
from sklearn.metrics import roc_auc_score, average_precision_score   # ranking metrics (explained below)
from torchvision import datasets              # downloads MNIST for us

## 1. Configuration
Change the scenario, data size, training, scoring, and output settings here.

In [12]:
# ---------- WHICH DIGITS ARE NORMAL / ANOMALOUS ----------
SCENARIO_TAG = "Train 0-6 inject 7-9"
NORMAL_DIGITS = [0, 1, 2, 3, 4, 5, 6]
ANOMALY_DIGITS = [7, 8, 9]

# ---------- DATA SIZE ----------
POOL_SIZE = 20000
INJECT_FRACTION = 0.05

# ---------- CLEANING SETTINGS ----------
CLEANING_ROUNDS = 3
REMOVE_FRACTION = 0.02
SCORE_MODE = "combined"
NUM_LATENT_SPACES = 5
NUM_SEEDS = NUM_LATENT_SPACES

# ---------- AUTOENCODER SETTINGS ----------
LATENT_DIM = 64
EPOCHS = 15
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
K_NEIGHBORS = 5

# ---------- OUTPUT ----------
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
OUTPUT_DIR = (Path("Iterations") / f"{NUM_LATENT_SPACES} Spaces"
              / f"{LATENT_DIM} Latent Dimensions")
RESULT_STEM = f"{SCENARIO_TAG} ({LATENT_DIM} Latent Dimensions) {RUN_ID}"
NUM_GRID_IMAGES = 40

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

USE_AMP = DEVICE.type == "cuda"
if USE_AMP:
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

assert not set(NORMAL_DIGITS) & set(ANOMALY_DIGITS), "digit lists must not overlap"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Device: {DEVICE} | normal {NORMAL_DIGITS} | anomalies {ANOMALY_DIGITS}")
if USE_AMP:
    print(f"CUDA GPU: {torch.cuda.get_device_name(0)} | mixed precision enabled")
print(f"Saving this run to: {OUTPUT_DIR} (run {RUN_ID})")

LOG_LINES = []
def log(text=""):
    """Print a line and remember it for the saved results file."""
    print(text)
    LOG_LINES.append(text)

Device: mps | normal [0, 1, 2, 3, 4, 5, 6] | anomalies [7, 8, 9]
Saving this run to: Iterations/5 Spaces/64 Latent Dimensions (run 20261110_211527_363387)


## 2. Autoencoder
The encoder maps each image to a `LATENT_DIM` vector. The decoder reconstructs the image.

In [13]:
class ConvAutoencoder(nn.Module):
    """Convolutional autoencoder for 28x28 grayscale images."""

    def __init__(self, latent_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64 * 7 * 7), nn.ReLU(),
            nn.Unflatten(1, (64, 7, 7)),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def train_autoencoder(images):
    """Train a fresh autoencoder, keeping the current training subset on the selected device."""
    model = ConvAutoencoder(LATENT_DIM).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    images = images.to(DEVICE)
    n = images.size(0)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    model.train()
    for _ in range(EPOCHS):
        permutation = torch.randperm(n, device=DEVICE)
        for start in range(0, n, BATCH_SIZE):
            batch = images[permutation[start:start + BATCH_SIZE]]
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                reconstruction, _ = model(batch)
            loss = F.mse_loss(reconstruction.float(), batch.float())
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
    return model

## 3. Cleaning signals

- Latent kNN distance: average distance to nearby training embeddings.
- Reconstruction error: squared pixel error divided by image ink.

Each signal is ranked, then combined according to `SCORE_MODE`.

In [14]:
@torch.no_grad()
def encode(model, images, batch_size=2048):
    """Encode images in chunks and return full-precision latent vectors on the CPU."""
    model.eval()
    encoded = []
    for start in range(0, images.size(0), batch_size):
        batch = images[start:start + batch_size].to(DEVICE)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            embeddings = model.encoder(batch)
        encoded.append(embeddings.float().cpu())
    return torch.cat(encoded)


@torch.no_grad()
def recon_error_per_ink(model, images, batch_size=2048):
    """Squared reconstruction error divided by each image's total ink."""
    model.eval()
    out = []
    for start in range(0, images.size(0), batch_size):
        batch = images[start:start + batch_size].to(DEVICE)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            reconstruction, _ = model(batch)
        reconstruction = reconstruction.float()
        squared_error = ((reconstruction - batch.float()) ** 2).sum(dim=(1, 2, 3))
        ink = batch.float().sum(dim=(1, 2, 3)) + 1e-6
        out.append((squared_error / ink).cpu())
    return torch.cat(out).numpy()


def knn_distance(query, reference, k, exclude_self=False):
    """Mean Euclidean distance to the k nearest reference embeddings."""
    query_device = query.to(DEVICE)
    reference_device = reference.to(DEVICE)
    out = []
    for start in range(0, query_device.size(0), 1000):
        distances = torch.cdist(query_device[start:start + 1000], reference_device)
        if exclude_self:
            nearest = distances.topk(k + 1, dim=1, largest=False).values[:, 1:]
        else:
            nearest = distances.topk(k, dim=1, largest=False).values
        out.append(nearest.mean(dim=1).cpu())
    return torch.cat(out).numpy()


def to_rank(values):
    """Convert scores to ranks in [0, 1]."""
    return np.argsort(np.argsort(values)) / (len(values) - 1)


def pick_score(knn, recon):
    """Combine the selected suspicion signals into one score."""
    if SCORE_MODE == "latent":
        return to_rank(knn)
    if SCORE_MODE == "recon":
        return to_rank(recon)
    return (to_rank(knn) + to_rank(recon)) / 2


def probe_predict(ref_lat, ref_labels, query_lat, k):
    """Classify query embeddings by majority vote among their nearest labelled references."""
    ref_lat_device = ref_lat.to(DEVICE)
    query_lat_device = query_lat.to(DEVICE)
    ref_labels_t = torch.from_numpy(ref_labels).long().to(DEVICE)
    preds = []
    for start in range(0, query_lat_device.size(0), 1000):
        distances = torch.cdist(query_lat_device[start:start + 1000], ref_lat_device)
        indices = distances.topk(k, dim=1, largest=False).indices
        votes = F.one_hot(ref_labels_t[indices], 10).sum(dim=1)
        preds.append(votes.argmax(dim=1).cpu())
    return torch.cat(preds).numpy()

## 4. Data
Load MNIST, select the configured normal and injected digits, and build fixed normal/anomaly test sets. The data loader searches this folder and its parents for `data/MNIST`.

In [15]:
def evaluate(model, train_imgs, train_labels):
    """Grade one trained model on normal classification and anomaly detection."""
    train_lat = encode(model, train_imgs)

    normal_lat = encode(model, ctx["normal_x"])
    preds = probe_predict(train_lat, train_labels, normal_lat, K_NEIGHBORS)
    acc = float((preds == ctx["normal_y"]).mean())

    det_lat = encode(model, ctx["det_x"])
    knn = knn_distance(det_lat, train_lat, K_NEIGHBORS)
    rec = recon_error_per_ink(model, ctx["det_x"])
    score = pick_score(knn, rec)
    y = ctx["det_is_anom"]
    return {
        "acc": acc,
        "auc": roc_auc_score(y, score),
        "ap": average_precision_score(y, score),
        "auc_latent": roc_auc_score(y, knn),
        "auc_recon": roc_auc_score(y, rec),
        "score": score,
    }

## 5. Evaluation
Accuracy uses a kNN classifier on normal test digits. AUROC and average precision use the selected novelty score on normal and anomaly test images.

In [16]:
def find_mnist_data_root():
    """Find MNIST under the project root even when the kernel starts in a subfolder."""
    working_dir = Path.cwd().resolve()
    required_raw_files = (
        "train-images-idx3-ubyte",
        "train-labels-idx1-ubyte",
        "t10k-images-idx3-ubyte",
        "t10k-labels-idx1-ubyte",
    )
    for base_dir in (working_dir, *working_dir.parents):
        data_root = base_dir / "data"
        mnist_dir = data_root / "MNIST"
        raw_dir = mnist_dir / "raw"
        processed_dir = mnist_dir / "processed"
        has_processed_data = (processed_dir / "training.pt").is_file() and (processed_dir / "test.pt").is_file()
        has_raw_data = raw_dir.is_dir() and all(
            (raw_dir / name).is_file() or (raw_dir / f"{name}.gz").is_file()
            for name in required_raw_files
        )
        if has_processed_data or has_raw_data:
            return data_root
    return working_dir / "data"


DATA_ROOT = find_mnist_data_root()
print(f"MNIST data root: {DATA_ROOT}")
train_set = datasets.MNIST(DATA_ROOT, train=True, download=True)
test_set = datasets.MNIST(DATA_ROOT, train=False, download=True)

train_x = train_set.data.float().div(255).unsqueeze(1)
train_y = train_set.targets.numpy()
test_x = test_set.data.float().div(255).unsqueeze(1)
test_y = test_set.targets.numpy()

normal_train_idx = np.where(np.isin(train_y, NORMAL_DIGITS))[0]
anomaly_train_idx = np.where(np.isin(train_y, ANOMALY_DIGITS))[0]
POOL_USED = min(POOL_SIZE, len(normal_train_idx))
N_INJECT = int(POOL_USED * INJECT_FRACTION / (1 - INJECT_FRACTION))
assert N_INJECT <= len(anomaly_train_idx), "not enough anomaly images to inject"

normal_mask = np.isin(test_y, NORMAL_DIGITS)
anomaly_mask = np.isin(test_y, ANOMALY_DIGITS)
ctx = {
    "normal_x": test_x[normal_mask],
    "normal_y": test_y[normal_mask],
    "det_x": torch.cat([test_x[normal_mask], test_x[anomaly_mask]]),
    "det_is_anom": np.concatenate([np.zeros(normal_mask.sum(), int),
                                   np.ones(anomaly_mask.sum(), int)]),
}

print(f"Normal training images available: {len(normal_train_idx)} | using {POOL_USED}")
print(f"Anomaly images injected per run: {N_INJECT} "
      f"({N_INJECT / (POOL_USED + N_INJECT) * 100:.1f}% of the training set)")
print(f"Test: {len(ctx['normal_y'])} normal images, {int(ctx['det_is_anom'].sum())} anomaly images")

MNIST data root: /Users/abhinavdamarla/Research/MNIST Latent Space Cleaning Verification copy/data
Normal training images available: 41935 | using 20000
Anomaly images injected per run: 1052 (5.0% of the training set)
Test: 6989 normal images, 3011 anomaly images


## 6. Experiment
For each seed, compare baseline training, iterative cleaning, one-shot removal, random removal, and a normal-only reference. Only the first three removal approaches share the same removal budget.

In [17]:
def run_seed(seed):
    rng = np.random.RandomState(seed)                                  # this seed's own random generator

    # ---- build the training set: real normal images + injected anomalies ----
    pool_idx = rng.choice(normal_train_idx, POOL_USED, replace=False)  # random normal images (no repeats)
    inj_idx = rng.choice(anomaly_train_idx, N_INJECT, replace=False)   # random anomaly images to inject
    all_idx = np.concatenate([pool_idx, inj_idx])
    all_x = train_x[all_idx]                                           # the images
    all_y = train_y[all_idx]                                           # their TRUE labels (grading only)
    is_inj = np.concatenate([np.zeros(len(pool_idx), bool),            # hidden ground truth: which are injected
                             np.ones(len(inj_idx), bool)])             # (grading only; the algorithm never sees it)
    n0 = len(all_x)
    per_round = int(n0 * REMOVE_FRACTION)                              # images removed each round
    total_remove = per_round * CLEANING_ROUNDS                         # total budget shared by every removal method
    log(f"\n===== seed {seed + 1}/{NUM_SEEDS} | {len(pool_idx)} normal + {N_INJECT} injected "
        f"| removing {per_round} per round ({total_remove} total) =====")

    def train_and_eval(indices, tag, offset):
        """Train a fresh autoencoder on all_x[indices] and grade it."""
        set_seed(seed * 100 + offset)                                  # different but repeatable seed per model
        model = train_autoencoder(all_x[indices])
        res = evaluate(model, all_x[indices], all_y[indices])
        log(f"  [{tag:<10}] n={len(indices):>6} | accuracy {res['acc'] * 100:5.2f}% "
            f"| detection AUROC {res['auc'] * 100:5.2f}%")
        return model, res

    def training_signals(model, indices):
        """Both suspicion signals for the images currently in the training set."""
        lat = encode(model, all_x[indices])
        knn = knn_distance(lat, lat, K_NEIGHBORS, exclude_self=True)   # self excluded: a point can't be its own neighbour
        rec = recon_error_per_ink(model, all_x[indices])
        return knn, rec

    out = {"n0": n0, "n_inj": N_INJECT}
    current = np.arange(n0)                                            # indices of images still in the training set
    hits_per_round = []                                                # injected images removed in each round
    round0 = None

    # ---- iterative cleaning loop ----
    for r in range(CLEANING_ROUNDS + 1):
        tag = "baseline" if r == 0 else f"round {r}"
        model, res = train_and_eval(current, tag, r)
        if r == 0:
            out["baseline"] = res                                      # round 0 = no cleaning yet
        if r == CLEANING_ROUNDS:                                       # final round: just grade, don't remove more
            out["iterative"] = res
            out["inj_left"] = int(is_inj[current].sum()) / len(current)
            break

        knn, rec = training_signals(model, current)
        scores = pick_score(knn, rec)                                  # suspicion score per training image
        if r == 0:
            round0 = (current.copy(), scores.copy())                   # remembered for the one-shot control
            out["signal_auc"] = (roc_auc_score(is_inj[current], knn),  # how well does each signal find injected images?
                                 roc_auc_score(is_inj[current], rec),
                                 roc_auc_score(is_inj[current], scores))
        worst = np.argsort(scores)[-per_round:]                        # positions of the highest-scoring (most suspicious) images
        removed = current[worst]                                       # their indices in all_x
        hits = int(is_inj[removed].sum())                              # how many were truly injected (grading only)
        hits_per_round.append(hits)
        log(f"      removed {per_round} | {hits} injected (precision {hits / per_round * 100:.1f}%, "
            f"chance would be {N_INJECT / n0 * 100:.1f}%)")
        if r == 0:
            top = np.argsort(scores)[::-1][:NUM_GRID_IMAGES]           # the most suspicious images, for the picture
            out["grid"] = [(all_x[current[p], 0].numpy(), int(all_y[current[p]]),
                            bool(is_inj[current[p]])) for p in top]
        current = np.delete(current, worst)                            # drop them from the training set

    out["hits_per_round"] = hits_per_round

    # ---- control: one-shot removal using ONLY the round-0 scores ----
    idx0, sc0 = round0
    keep = idx0[np.argsort(sc0)[:-total_remove]]                       # keep everything except the top total_remove scores
    out["oneshot"] = train_and_eval(keep, "one-shot", 90)[1]

    # ---- control: random removal of the same number of images ----
    keep = np.sort(rng.choice(n0, n0 - total_remove, replace=False))
    out["random"] = train_and_eval(keep, "random", 91)[1]

    # ---- ceiling: train on normal images only (no anomalies at all) ----
    out["ceiling"] = train_and_eval(np.arange(len(pool_idx)), "ceiling", 92)[1]   # first len(pool_idx) entries are the normal ones
    return out


## 7. Run
This fits seven models per seed. `NUM_LATENT_SPACES` controls the number of seeds. For a quick test, reduce it, `POOL_SIZE`, and `EPOCHS` in the configuration cell.

In [18]:
start_time = time.time()
log("=" * 70)
log(f"SCENARIO: Train on digits 0-6, inject 7-9  (latent dim {LATENT_DIM}, score mode {SCORE_MODE})")
log("=" * 70)

results = [run_seed(s) for s in range(NUM_SEEDS)]                      # one full experiment per seed

log(f"\n(finished in {(time.time() - start_time) / 60:.1f} minutes)")


SCENARIO: Train on digits 0-6, inject 7-9  (latent dim 64, score mode combined)

===== seed 1/5 | 20000 normal + 1052 injected | removing 421 per round (1263 total) =====


/var/folders/y4/fdlkj_2n7jdcn2d2snzttp040000gn/T/ipykernel_85655/4010246389.py:39: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
/var/folders/y4/fdlkj_2n7jdcn2d2snzttp040000gn/T/ipykernel_85655/4010246389.py:45: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/var/folders/y4/fdlkj_2n7jdcn2d2snzttp040000gn/T/ipykernel_85655/2172531513.py:8: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/var/folders/y4/fdlkj_2n7jdcn2d2snzttp040000gn/T/ipykernel_85655/2172531513.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp

  [baseline  ] n= 21052 | accuracy 98.28% | detection AUROC 66.43%


/var/folders/y4/fdlkj_2n7jdcn2d2snzttp040000gn/T/ipykernel_85655/2172531513.py:8: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/var/folders/y4/fdlkj_2n7jdcn2d2snzttp040000gn/T/ipykernel_85655/2172531513.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


      removed 421 | 63 injected (precision 15.0%, chance would be 5.0%)


/var/folders/y4/fdlkj_2n7jdcn2d2snzttp040000gn/T/ipykernel_85655/4010246389.py:39: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
/var/folders/y4/fdlkj_2n7jdcn2d2snzttp040000gn/T/ipykernel_85655/4010246389.py:45: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/var/folders/y4/fdlkj_2n7jdcn2d2snzttp040000gn/T/ipykernel_85655/2172531513.py:8: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/var/folders/y4/fdlkj_2n7jdcn2d2snzttp040000gn/T/ipykernel_85655/2172531513.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp

... (17 outputs left out)

/var/folders/y4/fdlkj_2n7jdcn2d2snzttp040000gn/T/ipykernel_85655/2172531513.py:8: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/var/folders/y4/fdlkj_2n7jdcn2d2snzttp040000gn/T/ipykernel_85655/2172531513.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


      removed 421 | 58 injected (precision 13.8%, chance would be 5.0%)


/var/folders/y4/fdlkj_2n7jdcn2d2snzttp040000gn/T/ipykernel_85655/4010246389.py:39: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
/var/folders/y4/fdlkj_2n7jdcn2d2snzttp040000gn/T/ipykernel_85655/4010246389.py:45: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


KeyboardInterrupt: 

## 8. Summary
Tables report mean and standard deviation across seeds. Paired differences compare methods on the same seed.

In [ ]:
def mean_std(values):
    v = np.array(values) * 100                                         # to percent
    return f"{v.mean():6.2f} +/- {v.std():4.2f}"

def paired(a_key, b_key, metric):
    """Per-seed difference between two methods (first minus second), in percentage points."""
    d = np.array([r[a_key][metric] - r[b_key][metric] for r in results]) * 100
    return d.mean(), d.std()

methods = [("Baseline (no cleaning)", "baseline"), ("Iterative cleaning", "iterative"),
           ("One-shot removal", "oneshot"), ("Random removal", "random"),
           ("Ceiling (no anomalies)", "ceiling")]

log("=" * 70)
log("RESULTS: Train on digits 0-6, inject 7-9")
log("=" * 70)
log(f"{'Method':<24} {'Accuracy %':>16} {'Detection AUROC %':>20} {'Avg precision %':>18}")
for name, key in methods:
    log(f"{name:<24} {mean_std([r[key]['acc'] for r in results]):>16} "
        f"{mean_std([r[key]['auc'] for r in results]):>20} "
        f"{mean_std([r[key]['ap'] for r in results]):>18}")

log("\nPaired differences in percentage points, mean (std). Positive = first method is better.")
for a, b in [("iterative", "baseline"), ("iterative", "oneshot"), ("iterative", "random")]:
    for metric, label in [("acc", "Accuracy"), ("auc", "Detection AUROC")]:
        m, s = paired(a, b, metric)
        log(f"  {label:<16} {a:>9} - {b:<9}: {m:+.2f} ({s:.2f})")

# ---- how well did the removal step find the injected images? ----
n0, n_inj = results[0]["n0"], results[0]["n_inj"]
log("\nREMOVAL QUALITY (did the suspicion score find the injected images?)")
prec = np.mean([r["hits_per_round"] for r in results], axis=0)         # average injected-hits per round across seeds
per_round = int(n0 * REMOVE_FRACTION)
log("  Precision by round (share of removed images that were injected): "
    + ", ".join(f"{h / per_round * 100:.1f}%" for h in prec)
    + f"   [random chance: {n_inj / n0 * 100:.1f}%]")
log(f"  Injected images found after all rounds: {prec.sum():.0f} of {n_inj} "
    f"({prec.sum() / n_inj * 100:.0f}%)")
log(f"  Injected share left in the training set: "
    f"{np.mean([r['inj_left'] for r in results]) * 100:.2f}% (started at {n_inj / n0 * 100:.2f}%)")
sig = np.array([r["signal_auc"] for r in results]) * 100
log(f"  Round-0 AUROC for finding injected TRAINING images -> latent distance {sig[:, 0].mean():.1f}%, "
    f"recon/ink {sig[:, 1].mean():.1f}%, combined {sig[:, 2].mean():.1f}%")
log("  (If these are near 50%, the injected images are not outliers: they form their own cluster.)")

# ---- plain-language verdict ----
log("\nVERDICT")
for metric, label in [("acc", "accuracy"), ("auc", "detection AUROC")]:
    gap = np.mean([r["ceiling"][metric] - r["baseline"][metric] for r in results]) * 100
    gain, gain_std = paired("iterative", "baseline", metric)
    vs_rand, rand_std = paired("iterative", "random", metric)
    vs_one, one_std = paired("iterative", "oneshot", metric)
    log(f"  {label[0].upper() + label[1:]}: anomalies cost {gap:+.2f} pp vs the ceiling. Iterative cleaning changed it by "
        f"{gain:+.2f} pp vs baseline, {vs_rand:+.2f} pp vs random, {vs_one:+.2f} pp vs one-shot.")
    if gap < 0.3:
        log(f"    -> The gap is tiny, so there is almost nothing for cleaning to recover on this metric.")
    elif gain > gain_std and vs_rand > rand_std:
        log(f"    -> Cleaning helped beyond what random removal explains (recovered {gain / gap * 100:.0f}% of the gap).")
    else:
        log(f"    -> No clear benefit over the controls (differences are within seed noise or negative).")
    if vs_one > one_std:
        log(f"    -> Iterating scored higher than one-shot removal by more than seed noise (only meaningful if cleaning also beat random).")
    else:
        log(f"    -> Iterating did NOT clearly beat one-shot removal.")


## 9. Plots
Accuracy and detection scores, novelty distributions, and examples removed in the first round.

In [ ]:
# ---------- (1) bar charts: accuracy and detection AUROC for every method ----------
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (metric, title) in zip(axes, [("acc", "Accuracy on normal test images (%)"),
                                      ("auc", "Anomaly detection AUROC (%)")]):
    means = [np.mean([r[k][metric] for r in results]) * 100 for _, k in methods]     # bar heights
    stds = [np.std([r[k][metric] for r in results]) * 100 for _, k in methods]       # error bars = seed-to-seed spread
    ax.bar(range(len(methods)), means, yerr=stds, capsize=4, color="#5b8def")
    ax.set_xticks(range(len(methods)))
    ax.set_xticklabels([n.split(" (")[0] for n, _ in methods], rotation=20, ha="right")
    lo = min(m - s for m, s in zip(means, stds))
    hi = max(m + s for m, s in zip(means, stds))
    ax.set_ylim(lo - (hi - lo) * 0.3, hi + (hi - lo) * 0.3)          # zoom the y-axis so small differences are visible
    ax.set_title(title)
fig.suptitle("Train on digits 0-6, inject 7-9  (error bars = std over seeds)")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / f"{RESULT_STEM} results.png", dpi=150)
plt.show()

# ---------- (2) novelty-score histograms for seed 1: baseline vs after cleaning ----------
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
y = ctx["det_is_anom"]
for ax, (key, title) in zip(axes, [("baseline", "Baseline (contaminated)"), ("iterative", "After iterative cleaning")]):
    s = results[0][key]["score"]
    ax.hist(s[y == 0], bins=40, alpha=0.6, density=True, label="normal test images")      # density=True makes areas comparable
    ax.hist(s[y == 1], bins=40, alpha=0.6, density=True, label="anomaly test images")
    ax.set_title(f"{title}  (AUROC {results[0][key]['auc'] * 100:.1f}%)")
    ax.set_xlabel("novelty score (rank)")
    ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / f"{RESULT_STEM} novelty histograms.png", dpi=150)
plt.show()

# ---------- (3) the images removed in round 1 (seed 1) ----------
grid = results[0]["grid"]
cols = 10
rows = math.ceil(len(grid) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.2, rows * 1.5))
flat = list(np.atleast_1d(axes).flat)
for ax in flat:
    ax.axis("off")                                                    # hide axes on every cell, including unused ones
for ax, (img, digit, injected) in zip(flat, grid):
    ax.imshow(img, cmap="gray")
    ax.set_title(("INJECTED " if injected else "normal ") + str(digit), fontsize=7,
                 color="red" if injected else "black")
fig.suptitle("Most suspicious training images (round 1): red = injected anomaly, black = real normal image")
fig.savefig(OUTPUT_DIR / f"{RESULT_STEM} removed images.png",
            dpi=150, bbox_inches="tight")
plt.show()

## 10. Save results

In [ ]:
out_path = OUTPUT_DIR / f"{RESULT_STEM}.txt"
out_path.write_text("\n".join(LOG_LINES), encoding="utf-8")            # everything passed to log() goes in the file
print(f"Saved results to: {out_path}")

## Notes

- Compare iterative cleaning with one-shot and random removal; similar scores mean the cleaning signal made little difference.
- AUROC near 50% means the selected score separates anomalies poorly.
- Results from few seeds can vary. Increase `NUM_LATENT_SPACES` for a steadier estimate.
- `SCORE_MODE` selects latent distance, reconstruction error, or both.